# Étape 5 — Test de la mémoire conversationnelle

**Objectif de ce notebook :**
Rejouer exactement la séquence de questions qui avait révélé 3 problèmes distincts
(voir le journal de développement), avec l'historique construit tour par tour comme
le fait l'application réelle, et vérifier automatiquement si chaque problème est
corrigé — pour un ou plusieurs modèles.

**Rappel des 3 problèmes détectés précédemment :**
- **A.** Le modèle inventait de fausses questions/réponses supplémentaires après sa
  réponse (continuation de dialogue non désirée).
- **B.** Le modèle recopiait la mise en forme du document source (numéros de section,
  titres de chapitre) au lieu de reformuler.
- **C.** L'historique de conversation "polluait" la réponse : une question sur un
  sujet différent du précédent réutilisait à tort les informations de l'échange
  d'avant (ex: répondre sur l'atelier Préparation à une question sur la Numérisation).

**Correctifs appliqués depuis :**
- Séquences d'arrêt (`stop`) sur le LLM → cible le problème A
- Nettoyage des en-têtes de chapitre à l'ingestion (`clean_text` dans `ingest.py`)
  → cible le problème B
- Renforcement du prompt (priorité stricte au contexte actuel, historique = référence
  uniquement) → cible le problème C

**Kernel à utiliser :** `Python (RAG GED)`.

## 0. Imports et accès au dossier `src/`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
sys.path.append(str(SRC_DIR))

from rag_chain import RagAssistant

## 1. Séquence de test (identique à celle qui posait problème)

On simule une vraie conversation : chaque question est posée avec l'historique des
échanges précédents, exactement comme le ferait Streamlit.

In [2]:
SEQUENCE_TEST = [
    "Comment fonctionne l'atelier Préparation ?",
    "Et pour la numérisation ?",
    "Quel logiciel est utilisé pour CMIM ?",
    "Et pour les autres clients ?",
]

## 2. Fonction de vérification automatique

Des contrôles simples, ciblés sur les 3 problèmes précédemment observés.

In [3]:
def verifier_reponse(indice_question: str, reponse: str) -> list[str]:
    """Retourne la liste des alertes détectées pour une réponse donnée."""
    alertes = []

    # Problème A : continuation de dialogue inventée
    marqueurs_continuation = ["Opérateur :", "Opérateur:", "\nQuestion :", "Question:"]
    if any(m in reponse for m in marqueurs_continuation):
        alertes.append("A - Continuation de dialogue détectée (le modèle invente un nouveau tour)")

    # Problème B : recopie de la mise en forme du document
    import re
    if re.search(r"\bCHAPITRE\s+\d", reponse) or re.search(r"\b\d\.\d\.\d\b", reponse):
        alertes.append("B - Recopie de numéros de section / titres de chapitre détectée")

    # Problème C : contenu hors-sujet lié à l'historique (vérif spécifique par question)
    if indice_question == "numerisation":
        if "agrafes" in reponse.lower() or "trombones" in reponse.lower():
            alertes.append("C - Réponse sur la numérisation contient des éléments de l'atelier Préparation")
        mots_numerisation = ["numéris", "code-barres", "codes-barres", "feuille séparatrice",
                              "feuilles séparatrices", "post-traitement", "orientation"]
        if not any(mot in reponse.lower() for mot in mots_numerisation):
            alertes.append("C - Réponse sur la numérisation ne contient aucun terme attendu du bon sujet")

    if indice_question == "cmim_logiciel":
        if "readsoft" in reponse.lower() and "kofax" in reponse.lower():
            alertes.append("Mélange détecté : les deux logiciels (Kofax Express et ReadSoft Forms) apparaissent ensemble")
        if "kofax express" not in reponse.lower():
            alertes.append("Logiciel attendu (Kofax Express) absent de la réponse")

    return alertes

## 3. Exécution de la séquence pour un modèle donné

In [4]:
def tester_memoire(nom_modele: str) -> None:
    print(f"\n{'#'*90}\n# MODÈLE : {nom_modele}\n{'#'*90}")
    assistant = RagAssistant(model=nom_modele)

    historique = []
    indices = ["preparation", "numerisation", "cmim_logiciel", "autres_clients"]

    for question, indice in zip(SEQUENCE_TEST, indices):
        resultat = assistant.ask(question, chat_history=historique)
        reponse = resultat["answer"]

        print(f"\n--- Question : {question} ---")
        print(reponse)

        alertes = verifier_reponse(indice, reponse)
        if alertes:
            for a in alertes:
                print(f"  ❌ {a}")
        else:
            print("  ✅ Aucun problème détecté")

        # Mise à jour de l'historique, comme le ferait l'application réelle
        historique.append({"role": "user", "content": question})
        historique.append({"role": "assistant", "content": reponse})

## 4. Test avec `llama3.2:3b`

In [5]:
tester_memoire("llama3.2:3b")


##########################################################################################
# MODÈLE : llama3.2:3b
##########################################################################################


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


--- Question : Comment fonctionne l'atelier Préparation ? ---
L'atelier Préparation consiste à :
1. Ouvrir chaque dossier du lot.
2. Classer les pièces à l'intérieur du dossier dans le bon ordre.
3. Vérifier la complétude du dossier (toutes les pièces attendues sont présentes).
4. Retirer les agrafes et trombones qui pourraient endommager le scanner.

Une erreur fréquente à ce stade est le classement accidentel d'une page dans le mauvais dossier. Cette erreur peut ensuite passer inaperçue lors du contrôle qualité manuel si elle n'est pas détectée automatiquement.
  ✅ Aucun problème détecté

--- Question : Et pour la numérisation ? ---
*   L'atelier Préparation est utilisé avant la numérisation des documents pour les préparer en les classant, en vérifiant leur complétude et en retirant les agrafes et trombones qui pourraient endommager le scanner.
*   Les opérations de post-traitement après la numérisation incluent la suppression des feuilles séparatrices, la suppression des pages blan

## 5. Test avec `llama3` (optionnel, pour comparaison)

⏱️ Plus lent — utile seulement si le test ci-dessus révèle encore des problèmes avec
`llama3.2:3b`, pour voir si le modèle plus gros s'en sort mieux sur la mémoire
conversationnelle spécifiquement.

In [ ]:
tester_memoire("llama3")


##########################################################################################
# MODÈLE : llama3
##########################################################################################


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


--- Question : Comment fonctionne l'atelier Préparation ? ---
L'atelier Préparation consiste à ouvrir chaque dossier du lot, le classer en ordre, vérifier la complétude des pièces et retirer les agrafes et trombones qui pourraient endommager le scanner.
  ✅ Aucun problème détecté

--- Question : Et pour la numérisation ? ---
La numérisation est une étape clé dans le processus de Gestion Électronique des Documents (GED) qui consiste à convertir les documents physiques en format électronique. Dans l'atelier Préparation, les opérations de post-traitement sont réalisées pour garantir la qualité des images numérisées, telles que la suppression des feuilles séparatrices, la suppression des pages blanches, la correction de l’orientation des documents, la vérification de la lisibilité des images et le contrôle de la lecture des codes-barres.
  ✅ Aucun problème détecté

--- Question : Quel logiciel est utilisé pour CMIM ? ---
Kofax Express.
  ✅ Aucun problème détecté

--- Question : Et pour le

: 

## Conclusion

- Si tous les ✅ apparaissent pour `llama3.2:3b` → les 3 problèmes sont résolus, pas
  besoin de changer de modèle, tu peux committer en confiance.
- Si des ❌ persistent avec `llama3.2:3b` mais disparaissent avec `llama3` → c'est une
  vraie limite du petit modèle sur la mémoire conversationnelle spécifiquement (à
  distinguer de la qualité générale testée dans le notebook précédent), et ça
  justifierait de garder `llama3` malgré la perte de vitesse.
- Si des ❌ persistent avec les deux modèles → le problème vient du prompt ou de la
  logique de récupération (retrieval), pas du modèle — il faudra qu'on retravaille
  `src/rag_chain.py`.